In [119]:
import numpy as np 
import pandas as pd
import joblib as jb
import streamlit as st
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,classification_report
from sklearn.linear_model import LogisticRegression

In [120]:
df = pd.read_csv("Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv")

In [121]:
df.head()

,transaction_id,user_id,age,gender,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,stress_level,academic_work_impact,addiction_level,addicted_label
0,TXN00001,U00001,21,Male,3.23,2.01,0.89,4.55,7.55,248,154,3.95,Medium,Yes,NaN,0
1,TXN00002,U00002,24,Other,5.09,3.81,2.24,4.44,7.66,127,71,6.71,Medium,Yes,NaN,0
2,TXN00003,U00003,31,Other,6.06,1.36,3.83,2.35,4.92,44,106,8.68,High,No,Mild,0
3,TXN00004,U00004,32,Other,7.83,5.85,1.51,3.54,8.23,178,107,9.77,High,Yes,Moderate,1
4,TXN00005,U00005,25,Male,9.96,5.92,3.42,5.27,6.21,136,177,12.55,Low,No,Severe,1


In [125]:
df.isnull().sum()

transaction_id             0
user_id                    0
age                        0
gender                     0
daily_screen_time_hours    0
social_media_hours         0
gaming_hours               0
work_study_hours           0
sleep_hours                0
notifications_per_day      0
app_opens_per_day          0
weekend_screen_time        0
stress_level               0
academic_work_impact       0
addiction_level            0
addicted_label             0
dtype: int64

In [124]:
df['addiction_level'].fillna(df['addiction_level'].mode()[0],inplace=True)

In [126]:
df.duplicated().sum()

np.int64(0)

In [127]:
df.dtypes

transaction_id              object
user_id                     object
age                          int64
gender                      object
daily_screen_time_hours    float64
social_media_hours         float64
gaming_hours               float64
work_study_hours           float64
sleep_hours                float64
notifications_per_day        int64
app_opens_per_day            int64
weekend_screen_time        float64
stress_level                object
academic_work_impact        object
addiction_level             object
addicted_label               int64
dtype: object

In [128]:
x = df.drop(['addicted_label','transaction_id','user_id','notifications_per_day','app_opens_per_day'],axis = 1)
y = df['addicted_label']

In [129]:
df['academic_work_impact'].value_counts()

academic_work_impact
No     3753
Yes    3747
Name: count, dtype: int64

In [130]:
df['addiction_level'].value_counts()

addiction_level
Moderate    3693
Severe      2434
Mild        1373
Name: count, dtype: int64

In [131]:
df['stress_level'].value_counts()

stress_level
High      2560
Low       2503
Medium    2437
Name: count, dtype: int64

In [132]:
numerical_cols = x.select_dtypes(include=['int64','float64']).columns.tolist()

In [133]:
categorical_cols = x.select_dtypes(include=['object']).columns.tolist()

In [134]:
numerical_transformer = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy = 'mean')),
    ('scaler',StandardScaler())
])

In [135]:
categorical_transformer = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy = 'most_frequent')),
    ('onehot',OneHotEncoder(handle_unknown = 'ignore',sparse_output = False))
     ])

In [136]:
preprocessor = ColumnTransformer(transformers=[
    ('num',numerical_transformer,numerical_cols),
    ('cat',categorical_transformer,categorical_cols)
])

In [137]:
X_train,X_test,y_train,y_test = train_test_split(x,y,test_size = 0.2,random_state = 42)

In [138]:
model = Pipeline(steps= [
                 ('pre',preprocessor),('reg',LogisticRegression(max_iter = 1000))
                 ])

In [139]:
model.fit(X_train,y_train)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age',
                                                   'daily_screen_time_hours',
                                                   'social_media_hours',
                                                   'gaming_hours',
                                                   'work_study_hours',
                                                   'sleep_hours',
                                                   'weekend_screen_time']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['gender', 'stress_level',
                                                   'academic_work_impact',
                                                   'addiction_level'])])),
                ('reg', LogisticRegression(max_iter=1000))])

In [140]:
y_pred = model.predict(X_test)
print(f'{classification_report(y_test,y_pred)}')

              precision    recall  f1-score   support

           0       0.96      0.96      0.96       456
           1       0.98      0.98      0.98      1044

    accuracy                           0.97      1500
   macro avg       0.97      0.97      0.97      1500
weighted avg       0.97      0.97      0.97      1500



In [141]:
jb.dump(model,'LogisticRegression.pkl')

['LogisticRegression.pkl']

In [156]:
load = jb.load('LogisticRegression.pkl')

st.title('Mobile Phone Usage Addiction Prediction')
age = st.number_input('age')
gender = st.selectbox('gender',['male','female','other'])
daily_screen_time_hours=st.number_input('daily_screen_time_hours')
social_media_hours = st.number_input('social_media_hours')

gaming_hours = st.number_input('gaming_hours')
work_study_hours = st.number_input('work_study_hours')
sleep_hours = st.number_input('sleep_hours')
weekend_screen_time = st.number_input('weekend_screen_time')
stress_level = st.selectbox('stress_level',['high','low','medium'])
academic_work_impact=st.selectbox('academic_work_impact',['no','yes'])
addiction_level = st.selectbox('addiction_level',['moderate','severe','mild'])
if st.button('predict'):
    data = pd.DataFrame({
                           
                            'age' :[age],                         
                         'gender' :[gender] ,                   
        'daily_screen_time_hours' :[daily_screen_time_hours], 
             'social_media_hours' :[social_media_hours] ,   
                   'gaming_hours' :[gaming_hours],         
               'work_study_hours' :[work_study_hours],        
                    'sleep_hours' :[sleep_hours],              
            'weekend_screen_time' :[weekend_screen_time],     
                   'stress_level' :[stress_level],              
           'academic_work_impact' :[academic_work_impact],    
                'addiction_level' :[addiction_level] 
    })
    prediction = load.predict(data)
    
    if prediction == 1:
        st.success('Addicted')
    else:
        st.success('Not Addicted')
    


2026-03-26 00:11:07.554 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-26 00:11:07.555 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-26 00:11:07.557 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-26 00:11:07.559 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-26 00:11:07.560 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-26 00:11:07.561 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-26 00:11:07.562 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-26 00:11:07.563 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar